# 02 — Fine-tuning view classifier (Tydzień 2)

**Cel sesji:** zbudować DataLoader na FETAL_PLANES_DB i dotrenować head klasyfikacyjny na zamrożonym backbone TinyUSFM.

**Definition of Done (z `Plan_Prac_VetEye_AI_USG.docx`):**
- ✅ walidacyjna accuracy **≥ 70%** na 6 klasach
- ✅ confusion matrix zapisana jako PNG
- ✅ checkpoint modelu na Google Drive

**Co robimy w tych komórkach:**
1. Pip install + mount Drive.
2. Wczytanie CSV z etykietami + train/val split (stratifikowany — równo rozkłada klasy).
3. PyTorch `Dataset` z augmentacjami pod USG (rotation, flip horyzontalny, brightness/contrast).
4. `DataLoader` z **WeightedRandomSampler** — żeby klasa "Other" (~31% datasetu) nie zalewała mniejszych klas.
5. Załadowanie TinyUSFM + zamrożenie backbone + dodanie głowy klasyfikacyjnej (6 klas).
6. Training loop, 5 epok, lr=1e-3 (AdamW). Checkpoint co epokę → Drive.
7. Walidacja: accuracy per epoch + finalny raport sklearn + confusion matrix PNG.

**Bezpiecznik (z planu):** jeśli accuracy < 50% po pełnym treningu — odmrażamy 2 ostatnie bloki backbone'u + lr=1e-4 (komórka na końcu). Jeśli to też nie pomoże — schodzimy na MobileNetV3 (osobny notebook 02b).

## 1. Setup: pip install, mount Drive, importy

**Co tu się dzieje:** instalujemy biblioteki ML, montujemy Google Drive, ustawiamy seed (żeby wyniki były powtarzalne — to ważne dla pracy końcowej).

In [ ]:
# Instalujemy: transformers (HF) do TinyUSFM, timm jako fallback, scikit-learn do confusion matrix,
# albumentations do augmentacji (lepsze niż torchvision dla obrazów medycznych)
!pip install -q transformers timm scikit-learn seaborn albumentations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Ustawienie seedów — żeby wyniki były takie same przy każdym uruchomieniu.
# Senior konsultanci to sprawdzą, więc lepiej żeby było reproducible.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Setup datasetu — rozpakowanie ZIP z Drive

Marto, masz `FETAL_PLANES_DB.zip` na Drive (link, który mi pokazałaś). **Robimy tu 3 rzeczy:**

1. **Sprawdzamy, że ZIP jest na Drive** — asercja: jak nie ma, kod krzyczy zamiast po cichu zawalić trening 20 minut później.
2. **Kopiujemy ZIP na lokalny dysk Colab (`/content`)** — Google Drive jest *bardzo* wolny przy odczycie 12 400 małych plików (każdy PNG ~50–200 kB). Lokalny dysk Colab czyta z prędkością SSD, więc trening jest ~10× szybszy. To standardowy trik dla datasetów w Colabie.
3. **Rozpakowujemy ZIP raz** i wykrywamy gdzie konkretnie wylądowały `Images/` i CSV (różne ZIP-y mają różną strukturę wewnętrzną — czasem jest jeszcze 1 folder na zewnątrz, czasem nie).

**⚠️ Ważne — lokalny dysk `/content` w Colabie jest kasowany przy każdej nowej sesji.** Czyli za każdym razem, gdy odpalasz notebook od zera, kroki 2 i 3 powtarzają się (~5 min total). To OK, bo ZIP zostaje bezpiecznie na Drive. **Checkpointy i wyniki zapisujemy NA DRIVE** (przeżyją reset sesji).

**Jeśli Twój ZIP nazywa się inaczej lub leży w podfolderze na Drive** — podmień tylko stałą `ZIP_ON_DRIVE` poniżej.

In [ ]:
import os, shutil, zipfile

# ============================================================
# KROK A: gdzie leży ZIP na Drive
# ============================================================
# UWAGA: jeśli wrzuciłaś ZIP do innego folderu/pod inną nazwą — podmień ścieżkę poniżej.
# Sprawdzić co masz: w komórce odpal: !ls "/content/drive/MyDrive/"
ZIP_ON_DRIVE = '/content/drive/MyDrive/FETAL_PLANES_DB.zip'

assert os.path.exists(ZIP_ON_DRIVE), (
    f'Nie znaleziono ZIP-a w: {ZIP_ON_DRIVE}\n'
    'Sprawdź gdzie jest: w nowej komórce odpal `!ls "/content/drive/MyDrive/"` '
    'i podmień ZIP_ON_DRIVE powyżej.'
)
size_mb = os.path.getsize(ZIP_ON_DRIVE) / (1024**2)
print(f'OK — ZIP znaleziony ({size_mb:.0f} MB)')

# ============================================================
# KROK B: kopia ZIP-a na lokalny dysk Colab + rozpakowanie
# ============================================================
# Idempotentne — jeśli już rozpakowane, pomija. Możesz odpalać tę komórkę wielokrotnie.
LOCAL_ZIP = '/content/FETAL_PLANES_DB.zip'
LOCAL_DATA_ROOT = '/content/FETAL_PLANES_DB'

if not os.path.exists(LOCAL_DATA_ROOT):
    if not os.path.exists(LOCAL_ZIP):
        print('Kopiuję ZIP z Drive na /content... (~30s — Drive jest wolny)')
        shutil.copy(ZIP_ON_DRIVE, LOCAL_ZIP)
        print('  -> skopiowane')
    print('Rozpakowuję ZIP... (~3–5 min dla 12 400 obrazów)')
    os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DATA_ROOT)
    print('  -> rozpakowane')
else:
    print(f'OK — już rozpakowane wcześniej, pomijam: {LOCAL_DATA_ROOT}')

# ============================================================
# KROK C: wykrycie struktury (gdzie konkretnie są Images/ i CSV)
# ============================================================
# `os.walk` chodzi po wszystkich podfolderach i szuka folderu o nazwie 'Images'.
# Bezpiecznie radzi sobie z różnymi strukturami ZIP-a (jeden folder zewnętrzny lub nie).
def find_images_folder(root):
    for dirpath, dirnames, _ in os.walk(root):
        if 'Images' in dirnames:
            return os.path.join(dirpath, 'Images')
    return None

IMG_DIR = find_images_folder(LOCAL_DATA_ROOT)
assert IMG_DIR is not None, (
    f'Nie znaleziono folderu Images/ pod {LOCAL_DATA_ROOT}.\n'
    f'Sprawdź w nowej komórce: !find {LOCAL_DATA_ROOT} -maxdepth 3 -type d'
)
DATA_DIR = os.path.dirname(IMG_DIR)  # folder zawierający Images/

# Szukamy CSV w tym samym folderze co Images/
csv_candidates = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.csv')]
assert csv_candidates, f'Nie znaleziono pliku CSV w {DATA_DIR}'
CSV_PATH = os.path.join(DATA_DIR, csv_candidates[0])

n_imgs = len(os.listdir(IMG_DIR))
print(f'\nDATA_DIR: {DATA_DIR}')
print(f'IMG_DIR:  {IMG_DIR}  ({n_imgs} plików — oczekiwane ~12 400)')
print(f'CSV_PATH: {CSV_PATH}')

# ============================================================
# KROK D: foldery na checkpointy i artefakty — ZOSTAJĄ NA DRIVE
# ============================================================
# (Przeżyją reset sesji Colab; lokalne /content zostanie skasowane!)
CKPT_DIR = '/content/drive/MyDrive/vet-eye-ai-usg-demo/checkpoints/02_finetune'
OUT_DIR  = '/content/drive/MyDrive/vet-eye-ai-usg-demo/outputs/02_finetune'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print(f'\nCheckpointy:  {CKPT_DIR}')
print(f'Artefakty:    {OUT_DIR}')

## 3. Wczytanie CSV + train/val split

FETAL_PLANES_DB ma kolumnę `Train` z autorskim splitem (1 = train, 0 = test).
**Co robimy:**
- używamy oryginalnego splitu testowego jako `val` (1796 obrazów wg paperu Burgos-Artizzu et al. 2020)
- patrzymy na rozkład klas (`Plane`) — z dużym prawdopodobieństwem zobaczymy nierównowagę ("Other" ~31%, "Fetal femur" ~6%), stąd później weighted sampler
- enkodujemy klasy jako int 0..5

In [ ]:
df = pd.read_csv(CSV_PATH, sep=';')  # FETAL_PLANES_DB używa średnika
print(f'Wczytano {len(df)} wierszy. Kolumny: {df.columns.tolist()}')
df.head()

In [ ]:
# Rozkład klas
print('Rozkład klas (Plane):')
print(df['Plane'].value_counts())
print()
print('Autorski split:')
print(df['Train '].value_counts() if 'Train ' in df.columns else df['Train'].value_counts())

In [ ]:
# UWAGA: w oryginalnym CSV kolumna nazywa się 'Train ' (ze spacją na końcu) — to nie błąd.
# Jeśli u Ciebie jest bez spacji, podmień poniżej.
TRAIN_COL = 'Train ' if 'Train ' in df.columns else 'Train'

# Mapowanie klasa -> int. Sortujemy alfabetycznie żeby było deterministyczne.
CLASSES = sorted(df['Plane'].unique().tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASSES)
print(f'{NUM_CLASSES} klas: {CLASSES}')

df['label'] = df['Plane'].map(CLASS_TO_IDX)
train_df = df[df[TRAIN_COL] == 1].reset_index(drop=True)
val_df   = df[df[TRAIN_COL] == 0].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

## 4. PyTorch Dataset + augmentacje

**Co robimy:**
- Resize do 224×224 (standard dla ViT — TinyUSFM też używa 224)
- Augmentacje **tylko w trainie**: rotacja ±15°, flip horyzontalny, lekka zmiana jasności/kontrastu. **Nie używamy flip vertical** ani agresywnych zniekształceń — w USG góra/dół ma znaczenie anatomiczne.
- Normalizacja: ImageNet mean/std (TinyUSFM trenowany na danych USG, ale weights inicjalizowane z ImageNet, więc ten preprocessing jest OK na start).

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0),
    A.Rotate(limit=15, p=0.5, border_mode=0),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class FetalPlanesDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Nazwa pliku w CSV bywa bez .png — dodajemy jeśli brakuje
        fname = row['Image_name']
        if not fname.lower().endswith('.png'):
            fname += '.png'
        path = os.path.join(self.img_dir, fname)
        # FETAL_PLANES_DB to obrazy grayscale w PNG — konwersja do RGB bo backbone oczekuje 3 kanałów
        img = np.array(Image.open(path).convert('RGB'))
        if self.transform is not None:
            img = self.transform(image=img)['image']
        label = int(row['label'])
        return img, label

train_ds = FetalPlanesDataset(train_df, IMG_DIR, transform=train_tf)
val_ds   = FetalPlanesDataset(val_df,   IMG_DIR, transform=val_tf)
print(f'train_ds: {len(train_ds)}, val_ds: {len(val_ds)}')
# Test — wczytaj pierwszy element
x, y = train_ds[0]
print(f'Tensor shape: {x.shape}, dtype: {x.dtype}, label: {y} ({IDX_TO_CLASS[y]})')

## 5. DataLoader + WeightedRandomSampler

**Dlaczego weighted sampler:** klasy są niezbalansowane. "Other" ma ~31% obrazów, "Fetal femur" ~6%. Bez balansowania model nauczy się przewidywać "Other" dla wszystkiego i będzie miał ~31% accuracy, ale 0% recall na małych klasach.

Każdej próbce nadajemy wagę odwrotnie proporcjonalną do liczności jej klasy → sampler losuje tak, żeby w batchu klasy były ~równie częste.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2  # Colab nie lubi większych wartości

# Wagi per klasa (inverse frequency)
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_df['label'].values]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
print(f'train batches: {len(train_loader)}, val batches: {len(val_loader)}')

## 6. Model: TinyUSFM (zamrożony) + head klasyfikacyjny

**Strategia:** zamrażamy cały backbone (gradient flow off → szybko, mało pamięci), dodajemy tylko nową głowę linear → 6 klas.

**⚠️ UWAGA dla Marty:** w komórce poniżej zakładam, że TinyUSFM ładuje się przez Hugging Face `transformers.AutoModel`. Jeśli w Tyg. 1 załadowałaś go inaczej (np. przez clone repo MacDunno/TinyUSFM + custom loader) — **podmień tylko fragment ładowania backbone**, reszta zostaje. Sprawdź też `embedding_dim` — dla ViT-Tiny to zwykle 192, dla ViT-Small 384. Po załadowaniu zrób `print(backbone)` i znajdź wymiar wyjścia.

In [ ]:
from transformers import AutoModel

# Ładujemy backbone — TinyUSFM. Jeśli nazwa repo na HF jest inna, podmień stringa.
BACKBONE_NAME = 'MacDunno/TinyUSFM'  # TODO: zweryfikuj że to jest checkpoint który załadowałaś w Tyg.1
backbone = AutoModel.from_pretrained(BACKBONE_NAME, trust_remote_code=True)

# Zamrażamy wszystkie parametry backbone'u
for p in backbone.parameters():
    p.requires_grad = False

# Wyciągamy wymiar embeddingu — wykrywamy go heurystycznie żeby kod był odporny
if hasattr(backbone, 'config') and hasattr(backbone.config, 'hidden_size'):
    EMB_DIM = backbone.config.hidden_size
elif hasattr(backbone, 'embed_dim'):
    EMB_DIM = backbone.embed_dim
else:
    # Fallback — przepuść 1 tensor i zmierz
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
        out = backbone(dummy)
        feat = out.last_hidden_state[:, 0] if hasattr(out, 'last_hidden_state') else out
        EMB_DIM = feat.shape[-1]
print(f'Embedding dim: {EMB_DIM}')

In [ ]:
class ViewClassifier(nn.Module):
    """Backbone (zamrożony) + linear head 6-klasowy."""
    def __init__(self, backbone, emb_dim, num_classes, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        out = self.backbone(x)
        # Dla ViT-owych modeli HF: bierzemy CLS token (pierwszy z last_hidden_state)
        if hasattr(out, 'last_hidden_state'):
            feat = out.last_hidden_state[:, 0]
        elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
            feat = out.pooler_output
        else:
            feat = out  # surowy tensor
        return self.head(self.dropout(feat))

model = ViewClassifier(backbone, EMB_DIM, NUM_CLASSES).to(DEVICE)

# Sanity check — ile parametrów trenowalnych?
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 7. Training loop

- 5 epok, AdamW, lr=1e-3 (zgodnie z Plan_Prac).
- Loss: `CrossEntropyLoss`.
- Po każdej epoce: walidacja + checkpoint na Drive (jakby Colab się rozłączył).
- Logi do listy → potem rysujemy krzywą.

Czas spodziewany na T4: ~3-5 min na epokę. Łącznie ~20-25 min.

In [ ]:
EPOCHS = 5
LR = 1e-3

criterion = nn.CrossEntropyLoss()
# Trenujemy tylko parametry HEADA (backbone zamrożony)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, losses = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            losses.append(loss.item())
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return np.mean(losses), accuracy_score(all_labels, all_preds), all_preds, all_labels

best_val_acc = 0.0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        if i % 50 == 0:
            print(f'  [Epoch {epoch} | batch {i}/{len(train_loader)}] loss={loss.item():.4f}')
    train_loss = np.mean(epoch_losses)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    elapsed = (time.time() - start) / 60
    print(f'Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | time={elapsed:.1f} min')

    # Checkpoint co epokę — żeby przerwana sesja Colab nie kasowała postępu
    ckpt_path = os.path.join(CKPT_DIR, f'epoch_{epoch:02d}_acc_{val_acc:.4f}.pt')
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'val_acc': val_acc,
        'classes': CLASSES,
    }, ckpt_path)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'val_acc': val_acc, 'classes': CLASSES,
        }, os.path.join(CKPT_DIR, 'best.pt'))
        print(f'  ↑ Nowy najlepszy model (val_acc={val_acc:.4f}) → best.pt')

print(f'\nKoniec treningu. Best val_acc: {best_val_acc:.4f}')
print(f'DoD spełnione: {"✅ TAK" if best_val_acc >= 0.70 else "❌ NIE — uruchom komórkę z bezpiecznikiem"}')

## 8. Walidacja: classification report + confusion matrix PNG

In [ ]:
# Ładujemy najlepszy checkpoint
best_ckpt = torch.load(os.path.join(CKPT_DIR, 'best.pt'), map_location=DEVICE)
model.load_state_dict(best_ckpt['model_state'])
print(f'Załadowano best.pt (epoch={best_ckpt["epoch"]}, val_acc={best_ckpt["val_acc"]:.4f})')

_, val_acc, preds, labels = evaluate(model, val_loader)
print(f'\nFinal val accuracy: {val_acc:.4f}\n')
print(classification_report(labels, preds, target_names=CLASSES, digits=4))

# Zapis raportu do JSON — przyda się do pracy końcowej
report_dict = classification_report(labels, preds, target_names=CLASSES, digits=4, output_dict=True)
with open(os.path.join(OUT_DIR, 'classification_report.json'), 'w') as f:
    json.dump({'val_acc': val_acc, 'report': report_dict, 'history': history}, f, indent=2)
print(f'Raport zapisany: {OUT_DIR}/classification_report.json')

In [ ]:
# Confusion matrix → PNG
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalized = recall per klasa

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Confusion matrix (counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Confusion matrix (row-normalized — recall per klasa)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
cm_path = os.path.join(OUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Zapisano: {cm_path}')

In [ ]:
# Krzywa uczenia
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs_x = range(1, EPOCHS + 1)
axes[0].plot(epochs_x, history['train_loss'], label='train_loss', marker='o')
axes[0].plot(epochs_x, history['val_loss'], label='val_loss', marker='o')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Loss')

axes[1].plot(epochs_x, history['val_acc'], label='val_acc', marker='o', color='green')
axes[1].axhline(0.70, color='red', linestyle='--', label='DoD = 0.70')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Validation accuracy')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. (Opcjonalna) Bezpiecznik — jeśli accuracy < 50%

Z Plan_Prac: "Model nie konwerguje → odmrozić ostatnie 2 bloki backbone'u, lr=1e-4".

Uruchom tę komórkę **tylko jeśli** powyższy trening dał < 50% val acc. Wtedy 2 ostatnie bloki transformera + head trenują się razem, lr=1e-4.

In [ ]:
# UWAGA: uruchamiaj świadomie. Zakomentowane na start.

# # Odmrażamy 2 ostatnie bloki transformera
# unfrozen_blocks = 2
# if hasattr(backbone, 'encoder') and hasattr(backbone.encoder, 'layer'):
#     layers = backbone.encoder.layer
# elif hasattr(backbone, 'blocks'):
#     layers = backbone.blocks  # timm-style
# else:
#     raise RuntimeError('Nie wiem jak nazywają się bloki tego backbone — sprawdź print(backbone).')
# for layer in layers[-unfrozen_blocks:]:
#     for p in layer.parameters():
#         p.requires_grad = True
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f'Po odmrożeniu: trainable={trainable:,}')

# # Nowy optimizer z mniejszym lr
# optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
# # → wróć do komórki training loop i odpal jeszcze raz (możesz zwiększyć EPOCHS do 8)

---

## Co dalej (Tydzień 3 — 18–24.05)

Według `Plan_Prac_VetEye_AI_USG.docx`:
- Iteracja accuracy (jeśli potrzeba: więcej epok, scheduler, mocniejsze augmentacje, ewentualnie pełny USFM zamiast TinyUSFM).
- Heurystyczny quality head — najprostsza wersja: gradient magnitude / variance Laplaciana jako proxy ostrości obrazu.
- Scripted instructions: mapa klasa-przewidywana → tekstowa instrukcja ("przesuń głowicę dogłowowo o 2 cm").
- Pełen pipeline: klip → klasa + jakość + instrukcja.

**Deliverable Tygodnia 2 — commit do repo:**
```
vet-eye-ai-usg-demo/
  notebooks/
    01_first_inference.ipynb
    02_finetune_view_classifier.ipynb   <- ten plik
  outputs/
    confusion_matrix.png
    learning_curves.png
    classification_report.json
```